In [1]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/distillation/mmlu_explained_answer_deepseek_v4_flash_extend_w_large.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/distillation/mmlu_explained_answer_deepseek_v4_flash_extend_w_large.parquet
Shape: (12032, 14)


,src,answer,options,category,question,cot_content,question_id,answer_index,total_tokens,meta_cluster,base_cluster,distill_ans_correct,distill_reasoning,distill_answer
0,ori_mmlu-jurisprudence,C,['There is no distinction between the two form...,law,Which of the following criticisms of Llewellyn...,NaN,1286,2,81,Legal Interpretation,Legal Theory Interpretations,True,Llewellyn's distinction between the grand and ...,c
1,ori_mmlu-international_law,E,"['Article 19', 'Article 11', 'Article 12', 'Ar...",law,Which of the following articles are not qualif...,NaN,1293,4,38,Legal Interpretation,Constitutional Law,True,To determine which articles are not qualified ...,e


## Accuracy


In [2]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 1.0000 (12032/12032)


## Missing distilled reasoning traces


In [3]:
missing_nan = int(df["distill_reasoning"].isna().sum())
empty_mask = df["distill_reasoning"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 0


## Reasoning trace length distribution (chars)


In [4]:
existing_reasoning = df.loc[~empty_mask, "distill_reasoning"]
existing_reasoning.str.len().describe()

count    12032.000000
mean      1768.702128
std       1523.789648
min          2.000000
25%        953.000000
50%       1571.000000
75%       2350.250000
max      83047.000000
Name: distill_reasoning, dtype: float64

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [5]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


n_opts_series = df["options"].apply(n_options)
valid_mask = pd.Series(
    [is_valid_answer(a, k) for a, k in zip(df["distill_answer"], n_opts_series)],
    index=df.index,
)
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 0 (0.00%)


### All invalid answers


In [6]:
invalid_answers = df.loc[~valid_mask, "distill_answer"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

### 10 shortest reasoning traces


In [7]:
shortest = existing_reasoning.str.len().nsmallest(10)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_reasoning"])
    print()

--- row 10595 (len=2) ---
Tc

--- row 1116 (len=9) ---
Answer: j

--- row 4 (len=108) ---
The tax paid is based on the assessed value, which is a percentage of the market value. The tax rate of 11.5

--- row 1881 (len=127) ---
The total time from the three shows is 3 × 30 = 90 minutes. Adding the 90-minute movie gives 90 + 90 = 180 minutes.  
Answer: d

--- row 1656 (len=131) ---
The markup is 40% of the selling price, so we compute 0.4 × $50.00 = $20.00. Therefore, the markup in dollars is $20.00.

Answer: g

--- row 88 (len=141) ---
The selling price is 85% of the cost of $4,200, which is calculated as 0.85 × $4,200 = $3,570. Therefore, the correct option is h.

Answer: h

--- row 3807 (len=177) ---
The unit cost is found by dividing the total price by the number of lemons. Since 5 lemons cost $2.00, the cost per lemon is $2.00 ÷ 5 = $0.40. This matches option g.

Answer: g

--- row 3557 (len=185) ---
To convert 48 meters to millimeters, recall that 1 meter equals 1000 millimeters. M

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [8]:
# invalid_idx = df.index[~valid_mask]
# df.loc[invalid_idx, "distill_reasoning"] = ""
# df.loc[invalid_idx, "distill_answer"] = ""
# df.loc[invalid_idx, "distill_ans_correct"] = False

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [ ]:
# short_mask = df["distill_reasoning"].fillna("").str.len() < 200
# short_idx = df.index[short_mask]
# df.loc[short_idx, "distill_reasoning"] = ""

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")

Reset 11 rows and wrote ../../data/out/distillation/mmlu_explained_answer_deepseek_v4_flash_extend_w_large.parquet
